# Inspecting the failure cases

The n=100 evaluation of `run5` gives mean per-case Dice of **0.747 (left)** and
**0.753 (right)**, but the median is ~0.789 and a handful of cases score near
zero. Those cases, not the average, are what stands between this pipeline and
the 0.82 / 0.75 of the prior paper: recovering the four worst left-gland cases
alone moves the left mean to roughly 0.78.

The pattern worth explaining is that **one gland fails completely while the
other is fine on the same scan** (`amos_0346`: left 0.000, right 0.770;
`amos_0356`: left 0.596, right 0.000). Image quality and patient anatomy are
shared between the two glands, so neither can be the cause. The remaining
candidates are:

| hypothesis | what you would see below |
| --- | --- |
| **annotation error** | the mask sits somewhere that is not an adrenal gland, or is drawn on the wrong side |
| **anatomical variant** | the gland is real but shaped/positioned unlike the training distribution |
| **outside the field of view** | the gland is cut off by the 384-pixel centre crop, so the model never saw it |
| **lateralisation error** | the model found the gland but wrote it into the *other* output channel |
| **genuine miss** | the model produced low probability everywhere near the gland |

This notebook separates those five. It reads `per_case.csv` to rank the
failures, audits each case's native geometry, overlays the ground truth on both
the native scan and the resampled volume the model actually saw, and then runs
the checkpoint on the same case so prediction and truth can be compared side by
side.

**It runs fine on CPU** — one case is a few seconds. No training happens here.

Run order: top to bottom. Only the first cell normally needs editing.

## 0. Configuration

In [ ]:
# Edit these if your layout differs, then run the notebook top to bottom.
from pathlib import Path

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_DIR    = REPO_ROOT / "runs" / "run5"            # holds best_model.pt + per_case.csv
DATA_ROOT  = REPO_ROOT.parent / "data" / "amos22"   # native NIfTI: imagesVa/, labelsVa/, ...
CACHE_DIR  = REPO_ROOT.parent / "cache"             # the same --cache-dir you trained with

CHECKPOINT = RUN_DIR / "best_model.pt"
PER_CASE   = RUN_DIR / "per_case.csv"               # written by scripts/evaluate_segmenter.py

# MIOpen builds a kernel database on the first convolution and the default
# location is read-only on the cluster (-> miopenStatusInternalError). This must
# run BEFORE torch is imported anywhere in the process.
import os, tempfile
for _var, _sub in (("MIOPEN_USER_DB_PATH", "miopen-db"), ("MIOPEN_CUSTOM_CACHE_DIR", "miopen-cache")):
    os.environ.setdefault(_var, os.path.join(tempfile.gettempdir(), _sub))
    os.makedirs(os.environ[_var], exist_ok=True)

for label, p in [("repo", REPO_ROOT), ("run dir", RUN_DIR), ("data root", DATA_ROOT),
                 ("cache dir", CACHE_DIR), ("checkpoint", CHECKPOINT), ("per_case.csv", PER_CASE)]:
    print(f"{'ok     ' if p.exists() else 'MISSING'}  {label:<12} {p}")

In [ ]:
import sys, csv, logging

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

# Import the preprocessing from the training script rather than reimplementing
# it. Anything that drifts between the two makes this notebook show you a
# volume the model never saw, which is worse than no notebook at all.
sys.path.insert(0, str(REPO_ROOT / "scripts"))
sys.path.insert(0, str(REPO_ROOT))

from train_adrenal_segmenter import (
    GeometryConfig, LEFT_CHANNEL_VALUE, RIGHT_CHANNEL_VALUE, THRESHOLDS,
    discover_cases, prepare_case,
)

COLOUR = {"left": "#ffb000", "right": "#00b4d8", "pred": "#ff2d95"}
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.titlesize": 9, "axes.labelsize": 8})

logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
logger = logging.getLogger("inspect")
print("imports ok")

## 1. Rank the failures

`per_case.csv` has one row per validation patient with a raw and a
post-processed Dice for each gland. A (case, gland) pair below `FAIL_BELOW` is
a failure worth looking at individually; everything above it is ordinary
variation and is not what this notebook is for.

In [ ]:
FAIL_BELOW = 0.50          # a (case, gland) pair under this is a failure, not variation
STAGE      = "postprocessed"   # "raw" or "postprocessed"

with PER_CASE.open(newline="", encoding="utf-8") as fh:
    per_case = list(csv.DictReader(fh))

GLANDS = [g for g in ("left", "right", "gland") if f"dice_{g}_{STAGE}" in per_case[0]]
print(f"{len(per_case)} cases, glands: {', '.join(GLANDS)}\n")

pairs = [
    {"case_id": r["case_id"], "gland": g,
     "dice": float(r[f"dice_{g}_{STAGE}"]),
     "dice_raw": float(r[f"dice_{g}_raw"]),
     "truth_voxels": int(r[f"truth_voxels_{g}"]),
     "components": int(r.get(f"components_{g}_before", 0) or 0)}
    for r in per_case for g in GLANDS
]
pairs.sort(key=lambda d: d["dice"])
failures = [p for p in pairs if p["dice"] < FAIL_BELOW]

scores = np.array([p["dice"] for p in pairs])
print(f"mean {scores.mean():.4f}   median {np.median(scores):.4f}   "
      f"min {scores.min():.4f}   below {FAIL_BELOW}: {len(failures)} of {len(pairs)} pairs "
      f"({100 * len(failures) / len(pairs):.0f}%)\n")

print(f"{'case':<12} {'gland':<6} {'dice':>7} {'raw':>7} {'truth vox':>10} {'components':>11}")
print("-" * 58)
for p in pairs[:15]:
    print(f"{p['case_id']:<12} {p['gland']:<6} {p['dice']:>7.3f} {p['dice_raw']:>7.3f} "
          f"{p['truth_voxels']:>10d} {p['components']:>11d}")

# How much of the deficit lives in the tail: what the mean would be if every
# failing pair were merely average.
for g in GLANDS:
    vals = np.array([p["dice"] for p in pairs if p["gland"] == g])
    fixed = np.where(vals < FAIL_BELOW, np.median(vals), vals)
    print(f"\n{g:<6} mean {vals.mean():.4f}  ->  {fixed.mean():.4f} if the "
          f"{int((vals < FAIL_BELOW).sum())} failing case(s) reached the median")

In [ ]:
# The cases this notebook will inspect. Leave FOCUS_CASES empty to take the
# worst ones automatically, or list case ids explicitly to look at something
# specific.
FOCUS_CASES = []          # e.g. ["amos_0346", "amos_0333", "amos_0356"]
N_AUTO      = 4

if not FOCUS_CASES:
    seen = []
    for p in failures:
        if p["case_id"] not in seen:
            seen.append(p["case_id"])
        if len(seen) >= N_AUTO:
            break
    FOCUS_CASES = seen

by_case = {r["case_id"]: r for r in per_case}
print("Inspecting:")
for cid in FOCUS_CASES:
    r = by_case[cid]
    detail = "   ".join(f"{g} {float(r[f'dice_{g}_{STAGE}']):.3f} "
                        f"({int(r[f'truth_voxels_{g}']):>5d} vox)" for g in GLANDS)
    print(f"  {cid:<12} {detail}")

## 2. What the checkpoint was trained with

Every geometry number below comes out of the checkpoint's own config, so the
notebook cannot disagree with the run it is inspecting.

In [ ]:
import torch

ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
cfg = ckpt.get("config", {})
THRESHOLD    = float(ckpt.get("threshold", ckpt.get("best_threshold", 0.5)))
SLICE_WINDOW = int(cfg.get("slice_window", 5))
COMBINED     = bool(cfg.get("combine_glands", False))
CHANNELS     = ("gland",) if COMBINED else ("left", "right")

GEOM = GeometryConfig(
    spacing_z=float(cfg.get("target_spacing_z", 2.5)),
    spacing_xy=float(cfg.get("target_spacing_xy", 1.0)),
    image_size=int(cfg.get("image_size", 384)),
    z_margin=int(cfg.get("z_margin", 32)),
    right_label=int(cfg.get("right_label", 11)),
    left_label=int(cfg.get("left_label", 12)),
)
CHANNEL_VALUE = {"left": LEFT_CHANNEL_VALUE, "right": RIGHT_CHANNEL_VALUE}


def truth_mask(coded, gland):
    """Boolean truth for one gland out of the label-coded cache mask."""
    if COMBINED or gland == "gland":
        return coded > 0
    return coded == CHANNEL_VALUE[gland]

print(f"epoch {ckpt.get('epoch')}  val dice {ckpt.get('val_dice')}  "
      f"operating threshold {THRESHOLD:.2f}")
print(f"channels {CHANNELS} | slice window {SLICE_WINDOW} | encoder {cfg.get('encoder')}")
print(f"geometry  {GEOM.spacing_xy} mm in-plane / {GEOM.spacing_z} mm z, "
      f"{GEOM.image_size}px centre crop, labels R={GEOM.right_label} L={GEOM.left_label}")
print(f"cache key {GEOM.cache_key()}")

# Rebuild the exact train/validation split so native file paths can be located.
from types import SimpleNamespace
_disc = SimpleNamespace(
    data_root=DATA_ROOT, seed=int(cfg.get("seed", 42)),
    val_fraction=float(cfg.get("val_fraction", 0.2)),
    max_train_cases=0, max_val_cases=0,
    modality=str(cfg.get("modality", "all")),
    mri_id_threshold=int(cfg.get("mri_id_threshold", 500)),
)
_train_records, _val_records = discover_cases(_disc, logger)
RECORDS = {r["case_id"]: r for r in _val_records + _train_records}
print(f"\nresolved {len(_val_records)} validation / {len(_train_records)} training case paths "
      f"(modality filter: {_disc.modality})")
missing = [c for c in FOCUS_CASES if c not in RECORDS]
print("NOT FOUND:", missing) if missing else print("all focus cases resolved")

## 3. Geometry audit — could the model have seen the gland at all?

Before looking at any picture, rule the cheap explanation in or out. Training
resamples to a fixed physical spacing and then takes a **centre crop of
`image_size` pixels**. A gland lying outside that crop is deleted before the
network ever runs, and no amount of tuning will recover it.

The audit maps each gland's native bounding box through the same resample and
crop, then compares the voxel count that survives into the cache against the
count expected from the resampling ratio alone. A retention well under 100%
with no other explanation means the crop ate part of the gland.

In [ ]:
def native_volume(case_id):
    """Native image and label as (Z, X, Y), plus voxel spacing — the same
    transpose prepare_case() applies, so indices are comparable."""
    rec = RECORDS[case_id]
    img_nii, lab_nii = nib.load(str(rec["image_path"])), nib.load(str(rec["label_path"]))
    image = np.transpose(np.asarray(img_nii.dataobj, dtype=np.float32), (2, 0, 1))
    label = np.transpose(np.asarray(lab_nii.dataobj, dtype=np.int16), (2, 0, 1))
    sx, sy, sz = (float(v) for v in img_nii.header.get_zooms()[:3])
    return image, label, (sz, sx, sy)


def gland_mask(label, gland):
    if gland == "gland":
        return (label == GEOM.left_label) | (label == GEOM.right_label)
    return label == (GEOM.left_label if gland == "left" else GEOM.right_label)


def crop_bounds(native_len, zoom, size):
    """Where the centre crop lands, expressed in NATIVE voxel indices."""
    resampled = int(round(native_len * zoom))
    if resampled <= size:                       # padded, nothing removed
        return 0.0, float(native_len), False
    start = (resampled - size) // 2
    return start / zoom, (start + size) / zoom, True


def audit(case_id):
    image, label, (sz, sx, sy) = native_volume(case_id)
    zoom = (sz / GEOM.spacing_z, sx / GEOM.spacing_xy, sy / GEOM.spacing_xy)
    voxel_mm3 = sx * sy * sz
    cached = load_cached(case_id)

    out = {"case_id": case_id, "shape": image.shape, "spacing": (sz, sx, sy),
           "zoom": zoom, "glands": {}}
    for gland in GLANDS:
        m = gland_mask(label, gland)
        n = int(m.sum())
        entry = {"native_voxels": n, "mm3": n * voxel_mm3}
        if n:
            zs, xs, ys = (np.flatnonzero(m.any(axis=tuple(a for a in (0, 1, 2) if a != ax)))
                          for ax in (0, 1, 2))
            entry["z_range"] = (int(zs.min()), int(zs.max()))
            entry["z_extent_mm"] = (zs.max() - zs.min() + 1) * sz
            entry["bbox_x"] = (int(xs.min()), int(xs.max()))
            entry["bbox_y"] = (int(ys.min()), int(ys.max()))

            inside = True
            for ax, (lo, hi) in ((1, entry["bbox_x"]), (2, entry["bbox_y"])):
                c0, c1, cropped = crop_bounds(image.shape[ax], zoom[ax], GEOM.image_size)
                if cropped and (lo < c0 or hi > c1):
                    inside = False
                entry[f"crop_{'x' if ax == 1 else 'y'}"] = (round(c0), round(c1))
            entry["inside_crop"] = inside

            if cached is not None:
                kept = int(truth_mask(cached["mask"], gland).sum())
                expected = n * zoom[0] * zoom[1] * zoom[2]
                entry["cached_voxels"] = kept
                entry["retention"] = kept / expected if expected else float("nan")
        out["glands"][gland] = entry
    return out


def load_cached(case_id):
    """The volume the model actually saw: from the training cache when present,
    otherwise rebuilt with the identical preprocessing."""
    path = CACHE_DIR / GEOM.cache_key() / f"{case_id}.npz"
    if path.exists():
        with np.load(path) as z:
            return {"case_id": case_id, "image": z["image"], "mask": z["mask"],
                    "n_positive": int(z["n_positive"]), "n_slices": int(z["n_slices"]),
                    "source": "cache"}
    case = prepare_case(RECORDS[case_id], GEOM)
    if case is not None:
        case["source"] = "rebuilt"
    return case


AUDITS = {}
for cid in FOCUS_CASES:
    AUDITS[cid] = audit(cid)

print(f"{'case':<12} {'gland':<6} {'dice':>6} {'native vox':>11} {'mm^3':>9} "
      f"{'z slices':>9} {'z mm':>7} {'in crop':>8} {'kept':>7}")
print("-" * 84)
for cid, a in AUDITS.items():
    for gland in GLANDS:
        e = a["glands"][gland]
        d = float(by_case[cid][f"dice_{gland}_{STAGE}"])
        if not e["native_voxels"]:
            print(f"{cid:<12} {gland:<6} {d:>6.3f} {'NO LABEL IN THIS SCAN':>50}")
            continue
        ret = e.get("retention")
        print(f"{cid:<12} {gland:<6} {d:>6.3f} {e['native_voxels']:>11d} {e['mm3']:>9.0f} "
              f"{e['z_range'][1] - e['z_range'][0] + 1:>9d} {e['z_extent_mm']:>7.1f} "
              f"{'yes' if e['inside_crop'] else 'NO':>8} "
              f"{('-' if ret is None else f'{ret:6.0%}'):>7}")
    print(f"{'':<12} shape {a['shape']}  spacing (z,x,y) "
          f"({a['spacing'][0]:.2f}, {a['spacing'][1]:.2f}, {a['spacing'][2]:.2f}) mm")

### Reading the audit

- **`in crop` = NO** → the gland is partly or wholly outside the 384-pixel window.
  Fix the field of view (larger `--image-size`, or crop around the body rather
  than the image centre), not the network.
- **`kept` far below 100%** with `in crop` = yes → the resampling is destroying a
  structure only a slice or two thick. `--target-spacing-z 1.5` is the test.
- **`native vox` in the low hundreds** → the gland is genuinely tiny in this scan;
  a few voxels of boundary error cost a lot of Dice, and this may be as good as
  it gets for that case.
- Everything normal → the explanation is in the pictures below.

## 4. Ground truth on the native scan

Is the annotation where an adrenal gland actually is? Amber is the left gland
(label 12), cyan the right (label 11). The dashed rectangle marks the centre
crop: anything outside it is invisible to the model.

In [ ]:
def window_ct(volume, lo=-135.0, hi=215.0):
    """HU window to [0, 1] for display. MRI has no HU scale, so fall back to a
    percentile window for case ids at or above the MRI threshold."""
    return (np.clip(volume, lo, hi) - lo) / max(hi - lo, 1e-6)


def display_window(image, case_id):
    digits = "".join(c for c in case_id if c.isdigit())
    is_mri = bool(digits) and int(digits) >= int(cfg.get("mri_id_threshold", 500))
    if is_mri:
        lo, hi = np.percentile(image, [1, 99])
        return (np.clip(image, lo, hi) - lo) / max(hi - lo, 1e-6)
    return window_ct(image)


def overlay(ax, base, masks, zoom_box=None):
    """masks: list of (boolean array, colour, alpha)."""
    ax.imshow(base, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    for m, colour, alpha in masks:
        if m is None or not m.any():
            continue
        rgba = np.zeros(m.shape + (4,), dtype=np.float32)
        rgb = tuple(int(colour[i:i + 2], 16) / 255 for i in (1, 3, 5))
        rgba[m] = (*rgb, alpha)
        ax.imshow(rgba, interpolation="nearest")
    if zoom_box:
        x0, x1, y0, y1 = zoom_box
        ax.set_xlim(y0, y1); ax.set_ylim(x1, x0)
    ax.set_xticks([]); ax.set_yticks([])


def show_native(case_id, gland=None, n_slices=6, zoom=True, pad=45):
    """Axial slices across a gland's native extent, ground truth overlaid."""
    image, label, (sz, sx, sy) = native_volume(case_id)
    a = AUDITS.get(case_id) or audit(case_id)
    glands = [gland] if gland else [g for g in GLANDS if a["glands"][g]["native_voxels"]]
    base = display_window(image, case_id)

    for g in glands:
        e = a["glands"][g]
        if not e["native_voxels"]:
            print(f"{case_id} {g}: no label in this scan"); continue
        z0, z1 = e["z_range"]
        zs = np.unique(np.linspace(z0, z1, n_slices).round().astype(int))
        box = None
        if zoom:
            box = (max(0, e["bbox_x"][0] - pad), min(image.shape[1], e["bbox_x"][1] + pad),
                   max(0, e["bbox_y"][0] - pad), min(image.shape[2], e["bbox_y"][1] + pad))

        fig, axes = plt.subplots(1, len(zs), figsize=(2.1 * len(zs), 2.5))
        axes = np.atleast_1d(axes)
        for ax, z in zip(axes, zs):
            overlay(ax, base[z],
                    [(label[z] == GEOM.left_label, COLOUR["left"], 0.75),
                     (label[z] == GEOM.right_label, COLOUR["right"], 0.75)],
                    zoom_box=box)
            ax.set_title(f"z={z}")
        d = float(by_case[case_id][f"dice_{g}_{STAGE}"]) if case_id in by_case else float("nan")
        fig.suptitle(f"{case_id} — {g} gland, truth only — Dice {d:.3f} — "
                     f"{e['native_voxels']} voxels over {z1 - z0 + 1} slices "
                     f"({e['z_extent_mm']:.0f} mm)", y=1.06, fontsize=10)
        plt.tight_layout(); plt.show()


show_native(FOCUS_CASES[0])

In [ ]:
# Whole-slice context for the same case: is the annotation on the anatomically
# correct side, and is anything else labelled oddly?
def show_context(case_id, gland=None, n_slices=3):
    show_native(case_id, gland=gland, n_slices=n_slices, zoom=False)


show_context(FOCUS_CASES[0])

## 5. The volume the model actually saw

Same case after resampling, HU windowing, normalisation and the centre crop —
i.e. the network's input, not the radiologist's view. If the gland looks
plausible on the native scan but has vanished or been mangled here, the
preprocessing is the problem.

In [ ]:
def show_cached(case_id, gland=None, n_slices=6, zoom=True, pad=45):
    case = load_cached(case_id)
    if case is None:
        print(f"{case_id}: prepare_case() returned None — no adrenal voxels survive preprocessing.")
        return
    image, coded = case["image"].astype(np.float32), case["mask"]
    glands = [gland] if gland else [g for g in GLANDS if truth_mask(coded, g).any()]
    lo, hi = np.percentile(image, [1, 99])
    base = (np.clip(image, lo, hi) - lo) / max(hi - lo, 1e-6)

    for g in glands:
        m = truth_mask(coded, g)
        if not m.any():
            print(f"{case_id} {g}: GONE after preprocessing (it exists in the native label)")
            continue
        zs_present = np.flatnonzero(m.any(axis=(1, 2)))
        zs = np.unique(np.linspace(zs_present.min(), zs_present.max(), n_slices).round().astype(int))
        xs, ys = np.flatnonzero(m.any(axis=(0, 2))), np.flatnonzero(m.any(axis=(0, 1)))
        box = ((max(0, xs.min() - pad), min(image.shape[1], xs.max() + pad),
                max(0, ys.min() - pad), min(image.shape[2], ys.max() + pad)) if zoom else None)

        fig, axes = plt.subplots(1, len(zs), figsize=(2.1 * len(zs), 2.5))
        axes = np.atleast_1d(axes)
        for ax, z in zip(axes, zs):
            overlay(ax, base[z], [(m[z], COLOUR[g], 0.75)], zoom_box=box)
            ax.set_title(f"z={z}")
        fig.suptitle(f"{case_id} — {g} gland as the model sees it ({case['source']}) — "
                     f"{int(m.sum())} voxels over {len(zs_present)} slices", y=1.06, fontsize=10)
        plt.tight_layout(); plt.show()


show_cached(FOCUS_CASES[0])

## 6. Run the checkpoint on the case

Loading the model separately from `evaluate_segmenter.py` keeps this notebook
independent of a job running on the cluster. The architecture is rebuilt from
the checkpoint's own config, so it always matches the weights.

In [ ]:
import segmentation_models_pytorch as smp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name=str(cfg.get("encoder", "resnet34")), encoder_weights=None,
    in_channels=SLICE_WINDOW, classes=len(CHANNELS),
    decoder_attention_type=None if cfg.get("decoder_attention") == "none" else "scse",
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"model on {DEVICE}")


@torch.no_grad()
def predict_volume(image, batch_size=8):
    """(Z, H, W) float volume -> (Z, n_channels, H, W) probabilities."""
    half, n = SLICE_WINDOW // 2, image.shape[0]
    out = np.zeros((n, len(CHANNELS)) + image.shape[1:], dtype=np.float32)
    for start in range(0, n, batch_size):
        centres = range(start, min(start + batch_size, n))
        windows = np.stack([image[np.clip(np.arange(c - half, c + half + 1), 0, n - 1)]
                            for c in centres])
        batch = torch.from_numpy(windows.astype(np.float32)).to(DEVICE)
        out[start:start + len(windows)] = torch.sigmoid(model(batch)).float().cpu().numpy()
    return out


def dice(pred, truth):
    pred, truth = pred.astype(bool), truth.astype(bool)
    denom = pred.sum() + truth.sum()
    return float("nan") if denom == 0 else 2.0 * np.logical_and(pred, truth).sum() / denom


PRED_CACHE = {}
def probabilities(case_id):
    if case_id not in PRED_CACHE:
        case = load_cached(case_id)
        PRED_CACHE[case_id] = (case, predict_volume(case["image"].astype(np.float32)))
    return PRED_CACHE[case_id]


for cid in FOCUS_CASES:
    probabilities(cid)
    print(f"  predicted {cid}")

## 7. Which failure mode is it?

This is the table that decides between the hypotheses. For each failing gland:

- **`max p in truth`** — the highest probability the model assigned anywhere
  inside the true gland. Near zero means the model saw nothing there at all;
  well above the threshold means it found the gland and the loss is elsewhere.
- **`best dice`** — the best achievable over the whole threshold sweep. If this
  is good and the operating-point Dice is not, the threshold is the problem,
  not the model.
- **`swapped dice`** — the prediction scored against the *other* gland's truth.
  A high value here is a lateralisation error: the gland was found and written
  to the wrong channel.
- **`centroid offset`** — distance in mm between predicted and true centres.
  Large means it segmented something else entirely (often the kidney or a
  vessel).

In [ ]:
def diagnose(case_id):
    case, probs = probabilities(case_id)
    coded = case["mask"]
    spacing = np.array([GEOM.spacing_z, GEOM.spacing_xy, GEOM.spacing_xy])
    rows = []
    for c, gland in enumerate(CHANNELS):
        truth = truth_mask(coded, gland)
        p = probs[:, c]
        pred = p >= THRESHOLD
        sweep = [dice(p >= t, truth) for t in THRESHOLDS]
        best_i = int(np.nanargmax(sweep)) if np.any(~np.isnan(sweep)) else 0

        row = {
            "gland": gland,
            "truth_voxels": int(truth.sum()),
            "pred_voxels": int(pred.sum()),
            "dice": dice(pred, truth),
            "best_dice": sweep[best_i],
            "best_threshold": float(THRESHOLDS[best_i]),
            "max_p_in_truth": float(p[truth].max()) if truth.any() else float("nan"),
            "mean_p_in_truth": float(p[truth].mean()) if truth.any() else float("nan"),
            "max_p_anywhere": float(p.max()),
        }
        if not COMBINED:
            other = truth_mask(coded, "right" if gland == "left" else "left")
            row["swapped_dice"] = dice(pred, other)
        if pred.any() and truth.any():
            cp = np.argwhere(pred).mean(axis=0) * spacing
            ct = np.argwhere(truth).mean(axis=0) * spacing
            row["centroid_mm"] = float(np.linalg.norm(cp - ct))
        else:
            row["centroid_mm"] = float("nan")
        rows.append(row)
    return rows


header = (f"{'case':<12} {'gland':<6} {'dice':>6} {'best':>6} {'@thr':>5} {'truth':>7} "
          f"{'pred':>7} {'max p in truth':>14} {'swap':>6} {'centroid mm':>12}")
print(header); print("-" * len(header))
DIAGNOSES = {}
for cid in FOCUS_CASES:
    DIAGNOSES[cid] = diagnose(cid)
    for r in DIAGNOSES[cid]:
        print(f"{cid:<12} {r['gland']:<6} {r['dice']:>6.3f} {r['best_dice']:>6.3f} "
              f"{r['best_threshold']:>5.2f} {r['truth_voxels']:>7d} {r['pred_voxels']:>7d} "
              f"{r['max_p_in_truth']:>14.3f} {r.get('swapped_dice', float('nan')):>6.3f} "
              f"{r['centroid_mm']:>12.1f}")

### Decision rule

| observation | conclusion | next step |
| --- | --- | --- |
| `max p in truth` < ~0.05 | the model saw nothing there — a genuine miss | look at the pictures in section 8; suspect an unusual appearance or an annotation on non-adrenal tissue |
| `max p in truth` high, `dice` low, `best dice` high | the operating threshold is wrong for this case | per-case or per-modality threshold, or better calibration |
| `swapped dice` high | lateralisation error | the two channels are being confused — check the label convention (R=11, L=12) and consider a laterality-aware loss |
| `centroid mm` large, `pred voxels` large | segmenting the wrong structure | connected-component pruning by position, or more hard negatives |
| `pred voxels` = 0 everywhere | the model is silent on this scan | usually pairs with a geometry or modality anomaly in section 3 |

## 8. Prediction against truth, slice by slice

Truth in amber/cyan, prediction in magenta, and the raw probability map so a
sub-threshold response is visible rather than invisible.

In [ ]:
def show_prediction(case_id, gland=None, n_slices=6, pad=45, prob_floor=0.02):
    case, probs = probabilities(case_id)
    coded = case["mask"]
    image = case["image"].astype(np.float32)
    lo, hi = np.percentile(image, [1, 99])
    base = (np.clip(image, lo, hi) - lo) / max(hi - lo, 1e-6)

    for c, g in enumerate(CHANNELS):
        if gland and g != gland:
            continue
        truth = truth_mask(coded, g)
        if not truth.any():
            continue
        p = probs[:, c]
        pred = p >= THRESHOLD
        zs_present = np.flatnonzero(truth.any(axis=(1, 2)))
        zs = np.unique(np.linspace(zs_present.min(), zs_present.max(), n_slices).round().astype(int))
        xs, ys = np.flatnonzero(truth.any(axis=(0, 2))), np.flatnonzero(truth.any(axis=(0, 1)))
        x0, x1 = max(0, xs.min() - pad), min(image.shape[1], xs.max() + pad)
        y0, y1 = max(0, ys.min() - pad), min(image.shape[2], ys.max() + pad)
        box = (x0, x1, y0, y1)

        fig, axes = plt.subplots(3, len(zs), figsize=(2.1 * len(zs), 6.6))
        axes = axes.reshape(3, -1)
        for j, z in enumerate(zs):
            overlay(axes[0, j], base[z], [(truth[z], COLOUR[g], 0.75)], zoom_box=box)
            overlay(axes[1, j], base[z], [(truth[z], COLOUR[g], 0.45),
                                          (pred[z], COLOUR["pred"], 0.55)], zoom_box=box)
            heat = np.ma.masked_less(p[z], prob_floor)
            axes[2, j].imshow(base[z], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            im = axes[2, j].imshow(heat, cmap="inferno", vmin=0, vmax=1, alpha=0.75,
                                   interpolation="nearest")
            axes[2, j].set_xlim(y0, y1); axes[2, j].set_ylim(x1, x0)
            axes[2, j].set_xticks([]); axes[2, j].set_yticks([])
            axes[0, j].set_title(f"z={z}   max p over slice {p[z].max():.2f}")
        for row, name in enumerate(("truth", "truth + prediction", "probability")):
            axes[row, 0].set_ylabel(name, fontsize=8)
        d = dice(pred, truth)
        fig.suptitle(f"{case_id} — {g} gland — Dice {d:.3f} at threshold {THRESHOLD:.2f} — "
                     f"max probability in truth {p[truth].max():.3f}", y=1.01, fontsize=10)
        plt.tight_layout(); plt.show()


show_prediction(FOCUS_CASES[0])

In [ ]:
# Every focus case in one sweep. Comment out if the notebook gets heavy.
for cid in FOCUS_CASES:
    failing = [r["gland"] for r in DIAGNOSES[cid] if r["dice"] < FAIL_BELOW]
    for g in failing or [r["gland"] for r in DIAGNOSES[cid]]:
        show_prediction(cid, gland=g, n_slices=5)

## 9. Coronal view

The adrenal glands sit as thin caps on the kidneys, and a coronal reslice shows
the craniocaudal relationship an axial montage hides — particularly useful for
judging whether an annotation is plausible or is actually on the kidney,
diaphragm or a vessel.

In [ ]:
def show_coronal(case_id, gland=None):
    case, probs = probabilities(case_id)
    coded, image = case["mask"], case["image"].astype(np.float32)
    lo, hi = np.percentile(image, [1, 99])
    base = (np.clip(image, lo, hi) - lo) / max(hi - lo, 1e-6)
    aspect = GEOM.spacing_z / GEOM.spacing_xy

    for c, g in enumerate(CHANNELS):
        if gland and g != gland:
            continue
        truth = truth_mask(coded, g)
        if not truth.any():
            continue
        pred = probs[:, c] >= THRESHOLD
        x = int(round(np.argwhere(truth)[:, 1].mean()))     # coronal plane through the gland
        ys = np.flatnonzero(truth.any(axis=(0, 1)))
        y0, y1 = max(0, ys.min() - 60), min(image.shape[2], ys.max() + 60)

        fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
        for ax, masks, title in (
            (axes[0], [(truth[:, x], COLOUR[g], 0.7)], "truth"),
            (axes[1], [(truth[:, x], COLOUR[g], 0.4), (pred[:, x], COLOUR["pred"], 0.55)],
             "truth + prediction"),
        ):
            ax.imshow(base[:, x], cmap="gray", vmin=0, vmax=1, aspect=aspect,
                      interpolation="nearest")
            for m, colour, alpha in masks:
                rgba = np.zeros(m.shape + (4,), dtype=np.float32)
                rgb = tuple(int(colour[i:i + 2], 16) / 255 for i in (1, 3, 5))
                rgba[m] = (*rgb, alpha)
                ax.imshow(rgba, aspect=aspect, interpolation="nearest")
            ax.set_xlim(y0, y1); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
        fig.suptitle(f"{case_id} — {g} gland, coronal reslice at x={x}", fontsize=10)
        plt.tight_layout(); plt.show()


for cid in FOCUS_CASES:
    show_coronal(cid)

## 10. A good case, for comparison

A failure only means something against a success from the same model. This
renders the best-scoring validation case with identical settings.

In [ ]:
best_case = max(per_case, key=lambda r: np.mean([float(r[f"dice_{g}_{STAGE}"]) for g in GLANDS]))
print(f"best case {best_case['case_id']}: " +
      "  ".join(f"{g} {float(best_case[f'dice_{g}_{STAGE}']):.3f}" for g in GLANDS))
show_prediction(best_case["case_id"], n_slices=5)

## 11. Record what you found

Fill this in as you go — the conclusions are what feed back into the next run,
and they are easy to lose between sessions.

| case | gland | Dice | what the images show | verdict | action |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

Verdicts worth using consistently: `annotation error`, `anatomical variant`,
`outside crop`, `lateralisation`, `threshold`, `genuine miss`, `too small`.

**How the verdict changes the next run:**

- *annotation error* on more than one or two cases → the ceiling is the labels,
  not the model. Report it; consider excluding the cases with a documented
  reason rather than quietly dropping them.
- *outside crop* → raise `--image-size`, or crop around the body centroid rather
  than the image centre.
- *too small / thin* → `--target-spacing-z 1.5`, which was already the planned
  run 6.
- *lateralisation* → the two-channel target is being confused; check the label
  convention and consider a loss term penalising cross-channel confusion.
- *threshold* → per-modality thresholds, or calibrate on the validation set.
- *genuine miss* with normal geometry → more capacity or a stronger encoder is
  the honest answer, and is the point at which the architecture review's
  alternatives (MedNeXt, a residual-encoder U-Net) become worth testing.